# MF2 — Resonance Parameters

Parsing, data access, DataFrame export, and round-trip serialization.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from kika.endf import read_endf, read_mf2, fetch_endf
from kika.endf.parsers.parse_mf2 import parse_mf2_mt151
from kika.endf.classes.mf2 import (
    UnresolvedCaseC, RMatrixLimited,
    URR_LValue_CaseC, URR_JState_CaseC, URR_EnergyPoint,
    RML_ParticlePair, RML_SpinGroup,
)

FE56 = '/home/MONLEON-JUAN/kika/kika/endf/files/n-026_Fe_056.endf'
O16  = '/home/MONLEON-JUAN/kika/kika/endf/files/n-008_O_016.endf'

## 1. Parse Fe-56 (LRU=1, LRF=3 — Reich-Moore)

In [2]:
fe56 = read_mf2(FE56)

assert fe56.formalism == 'Reich-Moore'
assert fe56.num_resonances == 320
assert fe56.num_l_values == 3
assert fe56.scattering_radius == 0.5444
assert fe56.target_spin == 0
assert fe56.energy_range == (1e-5, 850000)

print(fe56.summary())

MF2/MT151  ZA=26056  AWR=55.4544  NIS=1
  Isotope ZA=26056  ABN=1  LFW=0  NER=1
    Range [1e-05, 850000] eV  LRU=1 LRF=3 (Reich-Moore)
      SPI=0  AP=0.5444  NLS=3  NLSC=3
        l=0  NRS=40
        l=1  NRS=147
        l=2  NRS=133


## 2. Parse O-16 (LRU=0 — scattering radius only)

In [3]:
o16 = read_mf2(O16)

assert o16.formalism == 'none'
assert o16.num_resonances == 0
assert abs(o16.scattering_radius - 0.5494627) < 1e-6
assert o16.target_spin == 0

print(o16.summary())

MF2/MT151  ZA=8016  AWR=15.85751  NIS=1
  Isotope ZA=8016  ABN=1  LFW=0  NER=1
    Range [1e-05, 1.5e+08] eV  LRU=0 LRF=0 (none)
      SPI=0  AP=0.5494627000000001


## 3. Resonance DataFrame

In [4]:
df = fe56.resonances_dataframe()
assert df.shape[0] == 320
assert set(df.columns) == {'energy', 'spin', 'l', 'GN', 'GG', 'GFA', 'GFB'}

# Check per-l counts
assert (df['l'] == 0).sum() == 40
assert (df['l'] == 1).sum() == 147
assert (df['l'] == 2).sum() == 133

print(df.head(10))
print(f'\nShape: {df.shape}')

     energy  spin  l        GN       GG  GFA  GFB
0 -473000.0   0.5  0  308000.0  0.96000    0    0
1  -24000.0   0.5  0    2710.0  0.96000    0    0
2   -2440.0   0.5  0     193.0  0.86000    0    0
3   27791.0   0.5  0    1409.3  1.00050    0    0
4   74029.0   0.5  0     611.5  0.69561    0    0
5   83628.0   0.5  0    1215.1  0.52883    0    0
6  129861.0   0.5  0     588.0  0.62000    0    0
7  140479.0   0.5  0    2735.0  1.61000    0    0
8  169275.0   0.5  0     962.0  0.94000    0    0
9  187737.0   0.5  0    3620.0  1.05000    0    0

Shape: (320, 7)


## 4. Resonance energies

In [5]:
energies = fe56.resonance_energies()
assert len(energies) == 320
assert np.all(energies[:-1] <= energies[1:]), 'not sorted'

print(f'Min: {energies[0]:.1f} eV')
print(f'Max: {energies[-1]:.1f} eV')

Min: -473000.0 eV
Max: 1283000.0 eV


## 5. Spot-check specific resonance values

In [6]:
table = fe56.resonance_table()

# First l=0 resonance
l0_res = [r for r in table if r['l'] == 0]
first_l0 = min(l0_res, key=lambda r: abs(r['energy'] - (-473000)))
assert abs(first_l0['energy'] - (-473000)) < 1000
print(f"First l=0 resonance: ER={first_l0['energy']:.1f} eV, J={first_l0['spin']}")

# First l=1 resonance (approx ER=1151)
l1_res = [r for r in table if r['l'] == 1]
first_l1 = min(l1_res, key=lambda r: abs(r['energy']))
print(f"First l=1 resonance (by |ER|): ER={first_l1['energy']:.1f} eV, J={first_l1['spin']}")

First l=0 resonance: ER=-473000.0 eV, J=0.5
First l=1 resonance (by |ER|): ER=1151.0 eV, J=0.5


## 6. Round-trip: parse → str() → re-parse

In [ ]:
def round_trip_check(mt151, label):
    """Generic round-trip: serialize → re-parse → compare."""
    text = str(mt151)
    data_lines = [l for l in text.split('\n')
                  if len(l) >= 75 and l[72:75].strip() != '0']
    rt = parse_mf2_mt151(data_lines, 151)
    
    # Basic structure
    assert len(rt.isotopes) == len(mt151.isotopes), f'{label}: isotope count'
    
    for iso_orig, iso_rt in zip(mt151.isotopes, rt.isotopes):
        assert iso_orig.za == iso_rt.za
        assert iso_orig.lfw == iso_rt.lfw
        assert len(iso_orig.energy_ranges) == len(iso_rt.energy_ranges)
        
        for er_orig, er_rt in zip(iso_orig.energy_ranges, iso_rt.energy_ranges):
            assert er_orig.lru == er_rt.lru
            assert er_orig.lrf == er_rt.lrf
            assert type(er_orig.parameters) == type(er_rt.parameters), \
                f'{label}: {type(er_orig.parameters).__name__} vs {type(er_rt.parameters).__name__}'
    
    # Resolved resonances
    if mt151.num_resonances > 0:
        t1 = mt151.resonance_table()
        t2 = rt.resonance_table()
        if len(t1) == len(t2):
            for r1, r2 in zip(t1, t2):
                for k in r1:
                    assert r1[k] == r2[k], f'{label}: mismatch at {k}'
    
    print(f'{label} round-trip OK  ({mt151.num_resonances} resonances)')

round_trip_check(fe56, 'Fe-56')
round_trip_check(o16,  'O-16')

## 7. Integration: read_endf() includes MF2

In [8]:
endf = read_endf(FE56)
assert 2 in endf.mf
assert 151 in endf.mf[2].sections

mt151 = endf.mf[2][151]
assert mt151.num_resonances == 320
assert mt151.formalism == 'Reich-Moore'

print(f'MF sections in Fe-56: {sorted(endf.mf.keys())}')
print(f'MF2/MT151: {mt151.num_resonances} resonances, {mt151.formalism}')

/home/MONLEON-JUAN/kika/kika/endf/parsers/parse_endf.py:94: UserWarning: Skipping MF sections without parsers: [6, 12, 14, 33]. Only parsing: [1, 2, 3, 4, 34]
  warnings.warn(f"Skipping MF sections without parsers: {skipped_mfs}. Only parsing: {parseable_mfs}")


MF sections in Fe-56: [1, 2, 3, 4, 34]
MF2/MT151: 320 resonances, Reich-Moore


## 8. U-238: Resolved (LRF=3) + Unresolved Case C (LRF=2)

In [ ]:
u238 = fetch_endf('U238')
mf2_u238 = u238.mf[2][151]
print(mf2_u238.summary())

# Verify structure
assert len(mf2_u238.isotopes) == 1
iso = mf2_u238.isotopes[0]
assert len(iso.energy_ranges) == 2

# Range 1: Resolved (Reich-Moore)
er0 = iso.energy_ranges[0]
assert er0.lru == 1 and er0.lrf == 3

# Range 2: Unresolved Case C
er1 = iso.energy_ranges[1]
assert er1.lru == 2 and er1.lrf == 2
assert isinstance(er1.parameters, UnresolvedCaseC)
urr = er1.parameters
assert urr.nls == 3  # l=0,1,2
print(f'\nURR: NLS={urr.nls}, LSSF={urr.lssf}')
for lv in urr.l_values:
    for js in lv.j_states:
        print(f'  l={lv.l}, J={js.aj}: {len(js.energy_points)} energy points')

# Round-trip
round_trip_check(mf2_u238, 'U-238')

## 9. U-238 Unresolved DataFrame export

In [ ]:
df_urr = mf2_u238.unresolved_dataframe()
assert df_urr.shape[0] > 0
assert 'energy' in df_urr.columns
assert 'D' in df_urr.columns
assert 'GN0' in df_urr.columns

print(f'URR DataFrame shape: {df_urr.shape}')
print(f'Columns: {list(df_urr.columns)}')
print(f'L values: {sorted(df_urr["l"].unique())}')
print(f'J values: {sorted(df_urr["J"].unique())}')
print()
print(df_urr.head(10))

## 10. Pu-239: R-Matrix Limited (LRF=7) + Unresolved Case C

In [ ]:
pu239 = fetch_endf('Pu239')
mf2_pu239 = pu239.mf[2][151]
print(mf2_pu239.summary())

iso = mf2_pu239.isotopes[0]

# Range 1: R-Matrix Limited
er0 = iso.energy_ranges[0]
assert er0.lru == 1 and er0.lrf == 7
rml = er0.parameters
assert isinstance(rml, RMatrixLimited)
assert len(rml.particle_pairs) == 3
assert len(rml.spin_groups) == 2
assert rml.krm == 3  # R-Matrix

print(f'\nRML: {len(rml.particle_pairs)} particle pairs')
for i, pp in enumerate(rml.particle_pairs):
    print(f'  PP{i+1}: ZA_a={pp.za}, ZB_b={pp.zb}, MT={pp.mt}')
for i, sg in enumerate(rml.spin_groups):
    print(f'  SG{i+1}: J={sg.aj}, P={sg.pj}, NCH={len(sg.channels)}, NRS={len(sg.resonances)}')
print(f'  Total resonances: {sum(len(sg.resonances) for sg in rml.spin_groups)}')

# Range 2: Unresolved Case C
er1 = iso.energy_ranges[1]
assert er1.lru == 2 and er1.lrf == 2
assert isinstance(er1.parameters, UnresolvedCaseC)

# Round-trip
round_trip_check(mf2_pu239, 'Pu-239')

## 11. Cl-35: R-Matrix Limited (LRF=7, 8 spin groups)

In [ ]:
cl35 = fetch_endf('Cl35')
mf2_cl35 = cl35.mf[2][151]
print(mf2_cl35.summary())

iso = mf2_cl35.isotopes[0]
er = iso.energy_ranges[0]
assert er.lru == 1 and er.lrf == 7
rml = er.parameters
assert isinstance(rml, RMatrixLimited)
assert len(rml.particle_pairs) == 3
assert len(rml.spin_groups) == 8

nres_total = sum(len(sg.resonances) for sg in rml.spin_groups)
assert nres_total > 0
print(f'\nCl-35 RML: {nres_total} total resonances across {len(rml.spin_groups)} spin groups')

# Round-trip
round_trip_check(mf2_cl35, 'Cl-35')

## 12. U-235: Resolved (LRF=3) + Unresolved Case C (LFW=1)

In [ ]:
u235 = fetch_endf('U235')
mf2_u235 = u235.mf[2][151]
print(mf2_u235.summary())

iso = mf2_u235.isotopes[0]
assert iso.lfw == 1  # fissile
assert len(iso.energy_ranges) == 2

# Range 2: Unresolved
er1 = iso.energy_ranges[1]
assert er1.lru == 2 and er1.lrf == 2
urr = er1.parameters
assert isinstance(urr, UnresolvedCaseC)

# DataFrame
df_u235_urr = mf2_u235.unresolved_dataframe()
assert df_u235_urr.shape[0] > 0
print(f'\nU-235 URR DataFrame: {df_u235_urr.shape}')
print(df_u235_urr.head(5))

# Round-trip
round_trip_check(mf2_u235, 'U-235')

## 13. Deep round-trip: field-level comparison for URR and RML

In [ ]:
def deep_round_trip(mt151, label):
    """Field-level round-trip comparison for URR and RML."""
    text = str(mt151)
    data_lines = [l for l in text.split('\n')
                  if len(l) >= 75 and l[72:75].strip() != '0']
    rt = parse_mf2_mt151(data_lines, 151)
    
    for iso1, iso2 in zip(mt151.isotopes, rt.isotopes):
        for er1, er2 in zip(iso1.energy_ranges, iso2.energy_ranges):
            p1, p2 = er1.parameters, er2.parameters
            
            if isinstance(p1, UnresolvedCaseC):
                assert p1.spi == p2.spi and p1.ap == p2.ap and p1.lssf == p2.lssf
                for lv1, lv2 in zip(p1.l_values, p2.l_values):
                    assert lv1.l == lv2.l and lv1.awri == lv2.awri
                    for js1, js2 in zip(lv1.j_states, lv2.j_states):
                        assert js1.aj == js2.aj and js1.int_code == js2.int_code
                        assert js1.amun == js2.amun and js1.amug == js2.amug
                        for ep1, ep2 in zip(js1.energy_points, js2.energy_points):
                            for attr in ('es', 'd', 'gx', 'gn0', 'gg', 'gf'):
                                v1, v2 = getattr(ep1, attr), getattr(ep2, attr)
                                assert abs(v1 - v2) < 1e-5 * max(1, abs(v1)), \
                                    f'{label}: {attr} mismatch: {v1} vs {v2}'
            
            elif isinstance(p1, RMatrixLimited):
                assert p1.ifg == p2.ifg and p1.krm == p2.krm and p1.krl == p2.krl
                assert len(p1.particle_pairs) == len(p2.particle_pairs)
                for pp1, pp2 in zip(p1.particle_pairs, p2.particle_pairs):
                    assert pp1.mt == pp2.mt and pp1.za == pp2.za
                for sg1, sg2 in zip(p1.spin_groups, p2.spin_groups):
                    assert sg1.aj == sg2.aj and sg1.pj == sg2.pj
                    assert len(sg1.channels) == len(sg2.channels)
                    for ch1, ch2 in zip(sg1.channels, sg2.channels):
                        assert ch1.ipp == ch2.ipp and ch1.l == ch2.l
                    assert len(sg1.resonances) == len(sg2.resonances)
                    for r1, r2 in zip(sg1.resonances, sg2.resonances):
                        assert abs(r1.er - r2.er) < 1e-5 * max(1, abs(r1.er))
                        for w1, w2 in zip(r1.widths, r2.widths):
                            assert abs(w1 - w2) < 1e-5 * max(1, abs(w1))
    
    print(f'{label} deep round-trip OK')

deep_round_trip(mf2_u238, 'U-238')
deep_round_trip(mf2_u235, 'U-235')
deep_round_trip(mf2_pu239, 'Pu-239')
deep_round_trip(mf2_cl35, 'Cl-35')